# 1단계: 차량 탐지 모델 (COCO 사전학습 → AI-Hub 165 파인튜닝) — v2

**v1에서 v2로 바뀐 점:** 실제 라벨 json을 확인해보니 이미지 1장당 json 1개가 아니라,
**카메라(폴더) 1대당 json 1개**이며 그 안에 해당 카메라 이미지 전체(수만 장)에 대한
어노테이션이 들어있는 COCO 스타일 구조였습니다. 그래서 파일명 매칭 방식이 아니라
json 내부의 `file_name`을 기준으로 매칭하도록 전면 수정했습니다.

- 데이터: AI-Hub "교통문제 해결을 위한 CCTV 교통 영상(시내도로)" (dataSetSn=165)
- 클래스: 승용차, 소형버스, 대형버스, 트럭, 대형 트레일러, 오토바이(자전거), 보행자 (7종, `분류없음`은 제외)
- bbox 형식: `[xmin, ymin, xmax, ymax]` (픽셀 좌표)


## 0. 환경 확인 (GPU / 패키지)

In [1]:
import os, sys, json, shutil, random
from pathlib import Path
from collections import defaultdict

import cv2
import torch
from tqdm import tqdm

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA version: 12.8


In [2]:
# 처음 한 번만 실행 (이미 설치되어 있으면 생략 가능)
# !pip install ultralytics scikit-learn

from ultralytics import YOLO


## 1. 데이터 경로 설정

`교통안전(Bbox)`, `교통안전(Tracking)`처럼 **카테고리별로 이미지/라벨이 서로 다른
zip으로 나뉘어 다운로드된 경우**가 있어서, 경로를 리스트로 여러 개 지정할 수 있게
했습니다. 해당 없으면 리스트에 경로 하나만 넣으면 됩니다.

In [3]:
# ==== 실제 경로로 수정하세요 (여러 개면 리스트에 계속 추가) ====
IMAGE_ROOTS = [
    Path(r"C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training"),
]
LABEL_ROOTS = [
    Path(r"C:\Users\Win11Pro\Downloads\101.교통문제 해결을 위한 CCTV 교통 데이터(시내도로)\01.데이터\1.Training"),
]
# ================================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def find_all_files(roots, exts: set):
    files = []
    for root in roots:
        if not root.exists():
            print(f"[경고] 경로가 존재하지 않습니다: {root}")
            continue
        files.extend(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts)
    return files

image_files = find_all_files(IMAGE_ROOTS, IMAGE_EXTS)
label_json_files = find_all_files(LABEL_ROOTS, {".json"})

print(f"이미지 파일: {len(image_files):,}개")
print(f"라벨 json 파일: {len(label_json_files):,}개 (카메라 1대당 1개)")


이미지 파일: 456,584개
라벨 json 파일: 137개 (카메라 1대당 1개)


## 2. 라벨 인덱스 구축

json 137개(카메라 대수만큼)를 전부 읽어서, **카메라ID -> {이미지 파일명: [(category_id, bbox), ...]}**
형태의 인덱스를 미리 만들어둡니다. 이후 이미지 파일 하나하나를 열 때마다 이 인덱스에서
바로 조회만 하면 됩니다.

- 카메라ID는 json 안의 `meta[0]["camera_id"]`에서 가져옵니다 (파일명 파싱보다 안전).
- `category_id`가 8번(`분류없음`)인 박스는 제외합니다.
- 모든 json의 `categories`가 서로 같은지도 검증합니다 (다르면 경고 출력).

In [4]:
EXCLUDE_CATEGORY_NAMES = {"분류없음"}

# camera_id -> {file_basename: [(category_id, [x1,y1,x2,y2]), ...]}
label_index = {}

global_categories = None   # {id: name}
mismatch_warned = False

for jp in tqdm(label_json_files, desc="라벨 json 로딩중"):
    with open(jp, encoding="utf-8") as f:
        data = json.load(f)

    # 카테고리 일관성 체크
    cats = {c["id"]: c["name"] for c in data.get("categories", [])}
    if global_categories is None:
        global_categories = cats
    elif cats != global_categories and not mismatch_warned:
        print(f"[경고] 카테고리 구성이 다른 json 발견: {jp}")
        mismatch_warned = True

    # camera_id 추출
    meta_list = data.get("meta", [])
    camera_id = meta_list[0]["camera_id"] if meta_list else jp.stem.split("_")[-1]

    # image_id -> file basename
    image_id_to_name = {}
    for img in data.get("images", []):
        basename = img["file_name"].split("/")[-1]
        image_id_to_name[img["id"]] = basename

    cam_dict = label_index.setdefault(camera_id, {})

    for ann in data.get("annotations", []):
        basename = image_id_to_name.get(ann["image_id"])
        if basename is None:
            continue
        boxes = []
        for cat_id, bbox in zip(ann["category_id"], ann["bbox"]):
            cat_name = global_categories.get(cat_id, "")
            if cat_name in EXCLUDE_CATEGORY_NAMES:
                continue
            boxes.append((cat_id, bbox))
        if boxes:
            cam_dict[basename] = boxes

print(f"인덱싱된 카메라 수: {len(label_index)}")
print("카테고리:", global_categories)


라벨 json 로딩중: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 137/137 [00:43<00:00,  3.15it/s]

인덱싱된 카메라 수: 71
카테고리: {1: '승용차', 2: '소형버스', 3: '대형버스', 4: '트럭', 5: '대형 트레일러', 6: '오토바이(자전거)', 7: '보행자', 8: '분류없음'}


## 3. 클래스 매핑 (json에서 확인된 실제 카테고리 기준, `분류없음` 제외)

In [5]:
# 최종 YOLO 클래스 순서를 category_id 오름차순으로 고정 (분류없음 제외)
used_cat_ids = sorted(cid for cid in global_categories if global_categories[cid] not in EXCLUDE_CATEGORY_NAMES)

CAT_ID_TO_YOLO_ID = {cid: i for i, cid in enumerate(used_cat_ids)}
CLASS_NAMES = [global_categories[cid] for cid in used_cat_ids]

print("YOLO 클래스 순서:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")


YOLO 클래스 순서:
  0: 승용차
  1: 소형버스
  2: 대형버스
  3: 트럭
  4: 대형 트레일러
  5: 오토바이(자전거)
  6: 보행자


## 4. 이미지 ↔ 라벨 매칭

이미지가 속한 **폴더명을 카메라ID로 간주**하고, `label_index[camera_id][파일명]`으로
바로 조회합니다.

In [6]:
pairs = []          # (이미지 경로, [(category_id, bbox_xyxy), ...])
no_camera = []       # 이미지의 폴더명이 label_index에 아예 없는 경우
no_label_for_img = [] # 카메라는 있는데 그 이미지 파일명에 대한 라벨이 없는 경우

for img_path in tqdm(image_files, desc="매칭중"):
    camera_id = img_path.parent.name
    cam_dict = label_index.get(camera_id)
    if cam_dict is None:
        no_camera.append(img_path)
        continue
    boxes = cam_dict.get(img_path.name)
    if boxes is None:
        no_label_for_img.append(img_path)
        continue
    pairs.append((img_path, boxes))

print(f"매칭 성공: {len(pairs):,}개")
print(f"카메라ID 자체를 못 찾은 이미지: {len(no_camera):,}개")
print(f"카메라는 맞는데 해당 파일명 라벨이 없는 이미지: {len(no_label_for_img):,}개")

if no_camera:
    sample_missing_cams = sorted({p.parent.name for p in no_camera})[:10]
    print("라벨을 못 찾은 카메라ID 예시:", sample_missing_cams)


매칭중: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 456584/456584 [00:01<00:00, 357835.54it/s]

매칭 성공: 446,662개
카메라ID 자체를 못 찾은 이미지: 0개
카메라는 맞는데 해당 파일명 라벨이 없는 이미지: 9,922개


> ⚠️ `no_camera`가 많이 나온다면, `IMAGE_ROOTS`와 `LABEL_ROOTS`가 서로 다른
> 카테고리(예: 이미지는 Bbox+Tracking 둘 다 있는데 라벨은 Bbox만 받아둔 경우)를
> 가리키고 있을 가능성이 큽니다. 그 경우 누락된 카테고리의 라벨 zip을 추가로
> 받아서 `LABEL_ROOTS` 리스트에 경로를 하나 더 추가하면 됩니다.

## 5. YOLO 데이터셋 구조로 변환 (train/val 8:2 분리)

In [8]:
from sklearn.model_selection import train_test_split

OUTPUT_ROOT = Path("./yolo_dataset")
for split in ["train", "val"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

train_pairs, val_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
print(f"train: {len(train_pairs):,}개 / val: {len(val_pairs):,}개")


train: 357,329개 / val: 89,333개


In [10]:
def convert_and_save(pairs_subset, split, use_symlink=True):
    """use_symlink=True면 하드링크(os.link)로 연결해서 디스크 용량을 아낍니다.
    (윈도우는 os.symlink에 관리자 권한이 필요해서 symlink 대신 hardlink를 씁니다.)
    하드링크도 실패하면(드라이브가 다른 경우 등) 자동으로 복사로 넘어갑니다."""
    skipped = 0
    link_failed = 0
    for img_path, boxes in tqdm(pairs_subset, desc=f"{split} 변환중"):
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]
        yolo_lines = []
        for cat_id, bbox in boxes:
            if cat_id not in CAT_ID_TO_YOLO_ID:
                continue
            x1, y1, x2, y2 = bbox
            xc = (x1 + x2) / 2 / w
            yc = (y1 + y2) / 2 / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            # 좌표가 이미지 범위를 벗어나면 스킵 (라벨 오류 방지)
            if not (0 < xc < 1 and 0 < yc < 1 and 0 < bw <= 1 and 0 < bh <= 1):
                continue
            yolo_lines.append(f"{CAT_ID_TO_YOLO_ID[cat_id]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        if not yolo_lines:
            skipped += 1
            continue
        dst_img = OUTPUT_ROOT / "images" / split / img_path.name
        if not dst_img.exists():
            linked = False
            if use_symlink:
                try:
                    os.link(img_path.resolve(), dst_img)  # 하드링크 시도
                    linked = True
                except OSError:
                    link_failed += 1
            if not linked:
                shutil.copy2(img_path, dst_img)
        dst_lbl = OUTPUT_ROOT / "labels" / split / (img_path.stem + ".txt")
        with open(dst_lbl, "w") as f:
            f.write("\n".join(yolo_lines) + "\n")
    print(f"[{split}] 스킵된 샘플: {skipped}개")
    if link_failed:
        print(f"[{split}] 하드링크 실패로 복사 처리된 파일: {link_failed}개")

convert_and_save(train_pairs, "train")
convert_and_save(val_pairs, "val")

train 변환중: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 357329/357329 [4:15:01<00:00, 23.35it/s]


[train] 스킵된 샘플: 0개


val 변환중: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 89333/89333 [52:40<00:00, 28.26it/s]

[val] 스킵된 샘플: 0개


## 6. data.yaml 생성

In [4]:
yaml_lines = [
    f"path: {OUTPUT_ROOT.resolve()}",
    "train: images/train",
    "val: images/val",
    "",
    "names:",
]
for i, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {i}: {name}")

yaml_content = "\n".join(yaml_lines) + "\n"

data_yaml_path = OUTPUT_ROOT / "data.yaml"
with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content)


NameError: name 'OUTPUT_ROOT' is not defined

## 7. 학습 (COCO 사전학습 가중치 → 파인튜닝)

In [5]:
from pathlib import Path
data_yaml_path = Path("./yolo_dataset/data.yaml")   # 이미 변환된 데이터셋 경로를 직접 지정
print(data_yaml_path.exists())  # True가 나오면 정상

True


In [6]:
model = YOLO("yolov8s.pt")  # COCO 사전학습 가중치 (자동 다운로드)

results = model.train(
    data=str(data_yaml_path),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,          # GPU(cuda:0) 사용
    workers=8,
    project="flood_stage1",
    name="vehicle_detector",
    patience=10,
    exist_ok=True,
)



New https://pypi.org/project/ultralytics/8.4.87 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.86  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1

In [11]:
# 실제 저장 위치를 하드코딩하지 않고 결과 객체에서 바로 가져옴
save_dir = Path(results.save_dir)
best_model_path = save_dir / "weights" / "best.pt"
print("실제 저장 경로:", best_model_path, "| 존재 여부:", best_model_path.exists())

실제 저장 경로: C:\Project\PythonProject\PyTorch001\final_project\runs\detect\flood_stage1\vehicle_detector\weights\best.pt | 존재 여부: True


## 8. 검증

In [8]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)
print("클래스별 mAP50:")
for i, name in model.names.items():
    print(f"  {name}: {metrics.box.maps[i]:.4f}")

Ultralytics 8.4.86  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
val: Fast image access  (ping: 0.30.3 ms, read: 651.4352.9 MB/s, size: 791.2 KB)
val: Scanning C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset\labels\val.cache... 85933 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 85934/85934  0.0s
val: C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset\images\val\fps_20_20201115_140000_04_1165.jpg: ignoring corrupt image/label: broken data stream when reading image file
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5371/5371 4.8it/s 18:440.2ss
                   all      85933     810031       0.77      0.589      0.648      0.506
                         85110     614409      0.897      0.891      0.919      0.759
                         1579       1622      0.637      0.581      0.558      0.481
                        27388      37702   

In [ ]:
import shutil
FINAL_MODEL_PATH = Path("./models/stage1_vehicle_detector.pt")
FINAL_MODEL_PATH.parent.mkdir(exist_ok=True)
shutil.copy2(best_model_path, FINAL_MODEL_PATH)
print("최종 모델 저장 위치:", FINAL_MODEL_PATH.resolve())

## 9. 학습된 모델로 빠른 추론 테스트

In [13]:
from pathlib import Path
import random

trained_model = YOLO(best_model_path)

# val_pairs 대신 디스크에 이미 변환되어 있는 val 이미지 중 하나를 직접 사용
val_img_dir = Path("./yolo_dataset/images/val")
val_images = list(val_img_dir.glob("*.jpg"))
print(f"val 이미지 {len(val_images):,}개 중 하나로 테스트")

sample_test_img = str(random.choice(val_images))
result = trained_model.predict(sample_test_img, save=True, conf=0.25)
print("테스트 이미지:", sample_test_img)
print("결과 저장 위치:", result[0].save_dir)

val 이미지 85,934개 중 하나로 테스트

image 1/1 C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset\images\val\fps_20_20201115_060000_04_0141.jpg: 384x640 4 s, 161.7ms
Speed: 4.2ms preprocess, 161.7ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Project\PythonProject\PyTorch001\final_project\runs\detect\predict
테스트 이미지: yolo_dataset\images\val\fps_20_20201115_060000_04_0141.jpg
결과 저장 위치: C:\Project\PythonProject\PyTorch001\final_project\runs\detect\predict


# 10. 테스트

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

# best_model_path 변수가 없으면(커널 재시작 등) 디스크에서 자동으로 찾음
try:
    best_model_path
except NameError:
    candidates = list(Path(".").rglob("best.pt"))
    best_model_path = candidates[0]
    print("best.pt 자동 탐색:", best_model_path)

trained_model = YOLO(best_model_path)
print("모델 로드 완료:", best_model_path)
print("클래스:", trained_model.names)

In [14]:
# 한 장

def detect_and_show(image_path, conf=0.25, figsize=(12, 8)):
    result = trained_model.predict(str(image_path), conf=conf, verbose=False)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)  # BGR -> RGB

    plt.figure(figsize=figsize)
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(Path(image_path).name)
    plt.show()

    print(f"탐지된 객체 수: {len(result.boxes)}")
    for box in result.boxes:
        cls_name = trained_model.names[int(box.cls[0])]
        conf_score = float(box.conf[0])
        print(f"  - {cls_name}: {conf_score:.2f}")

# 본인 컴퓨터에 있는 아무 사진 경로로 바꿔서 테스트
test_image_path = r"C:\Users\Win11Pro\Downloads\test_photo.jpg"
detect_and_show(test_image_path)

FileNotFoundError: C:\Users\Win11Pro\Downloads\test_photo.jpg does not exist

In [15]:
# 경로 지정(여러장) 

def detect_folder(folder_path, conf=0.25, max_show=6):
    folder = Path(folder_path)
    img_paths = [p for p in folder.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    print(f"{len(img_paths):,}개 이미지 발견")

    results = trained_model.predict([str(p) for p in img_paths], conf=conf, save=True, verbose=False)
    print("결과 저장 위치:", results[0].save_dir)

    # 처음 몇 장만 화면에 미리보기 (전체 다 띄우면 노트북이 무거워짐)
    for r in results[:max_show]:
        annotated = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(annotated)
        plt.axis("off")
        plt.title(Path(r.path).name)
        plt.show()

test_folder_path = r"C:\Users\Win11Pro\Downloads\test_images"
detect_folder(test_folder_path)

FileNotFoundError: C:\Users\Win11Pro\Downloads\test_photo.jpg does not exist

In [ ]:
# 영상파일 

def detect_video(video_path, conf=0.25, preview_every_n_sec=5):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    preview_interval = max(int(fps * preview_every_n_sec), 1)

    result_gen = trained_model.predict(
        source=str(video_path),
        conf=conf,
        save=True,     # 탐지 결과가 그려진 영상 파일로 저장됨
        stream=True,   # 프레임 단위로 처리해서 메모리 절약
    )

    saved_dir = None
    for i, r in enumerate(tqdm(result_gen, desc="영상 처리중")):
        if saved_dir is None:
            saved_dir = r.save_dir
        if i % preview_interval == 0:
            annotated = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 6))
            plt.imshow(annotated)
            plt.axis("off")
            plt.title(f"frame {i}")
            plt.show()

    print("결과 영상 저장 위치:", saved_dir)

test_video_path = r"C:\Users\Win11Pro\Downloads\test_video.mp4"
detect_video(test_video_path)

In [16]:
# 실시간

def detect_webcam(conf=0.25, cam_index=0):
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened():
        print("웹캠을 열 수 없습니다.")
        return
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        result = trained_model.predict(frame, conf=conf, verbose=False)[0]
        cv2.imshow("YOLO 실시간 탐지 (q: 종료)", result.plot())
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

# detect_webcam()  # 실행하려면 주석 해제